In [17]:
import polars as pl
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,accuracy_score

In [6]:
data_path = "../data/raw/Phones_accelerometer.csv"

if os.path.exists(data_path):
    print("File found! Creating a lazy execution plan...")

    lazy_df = pl.scan_csv(data_path)
    query_plan = lazy_df.head(5)

    actual_df = query_plan.collect()
    print("First 5 rows successfully loaded into memory:")
    display(actual_df)
else:
    print(f"Error: Could not find the file at {data_path}.")

File found! Creating a lazy execution plan...
First 5 rows successfully loaded into memory:


Index,Arrival_Time,Creation_Time,x,y,z,User,Model,Device,gt
i64,i64,i64,f64,f64,f64,str,str,str,str
0,1424696633908,1424696631913248572,-5.958191,0.6880646,8.135345,"""a""","""nexus4""","""nexus4_1""","""stand"""
1,1424696633909,1424696631918283972,-5.95224,0.6702118,8.136536,"""a""","""nexus4""","""nexus4_1""","""stand"""
2,1424696633918,1424696631923288855,-5.995087,0.653549,8.204376,"""a""","""nexus4""","""nexus4_1""","""stand"""
3,1424696633919,1424696631928385290,-5.942718,0.676163,8.128204,"""a""","""nexus4""","""nexus4_1""","""stand"""
4,1424696633929,1424696631933420691,-5.991516,0.641647,8.135345,"""a""","""nexus4""","""nexus4_1""","""stand"""


In [7]:
lazy_df = pl.scan_csv(data_path)

query_plan = lazy_df.select([
    pl.col("User").n_unique().alias("Total Unique Users"),
    pl.col("Model").n_unique().alias("Total Phone Models"),
    pl.col("Device").n_unique().alias("Total Specific Devices"),
    pl.col("gt").n_unique().alias("Total Activities")
])

summary_df = query_plan.collect()
display(summary_df)

activities = lazy_df.select(pl.col('gt').unique().drop_nulls()).collect()
print("\nUnique Activities:")
print(activities["gt"].to_list())

Total Unique Users,Total Phone Models,Total Specific Devices,Total Activities
u32,u32,u32,u32
9,4,8,7



Unique Activities:
['sit', 'bike', 'stand', 'null', 'walk', 'stairsup', 'stairsdown']


In [8]:
lazy_df = pl.scan_csv(data_path)

audit_plan = lazy_df.select([
    pl.len().alias('total rows'),
    pl.col('gt').is_null().sum().alias('target_is_null_count'),
    (pl.col('gt') == "null").sum().alias("target_string_null_count"),
    pl.col("x").is_null().sum().alias("x_nulls"),
    pl.col("y").is_null().sum().alias("y_nulls"),
    pl.col("z").is_null().sum().alias("z_nulls")
])

audit_results = audit_plan.collect()
display(audit_results)

valid_activity_plan = (
    lazy_df
    .filter(pl.col('gt').is_not_null() & (pl.col("gt") != "null"))
    .group_by("gt")
    .agg(pl.len().alias("count"))
    .sort("count",descending=True) 
)

activity_distribution = valid_activity_plan.collect()
display(activity_distribution)


total rows,target_is_null_count,target_string_null_count,x_nulls,y_nulls,z_nulls
u32,u32,u32,u32,u32,u32
13062475,0,1783200,0,0,0


gt,count
str,u32
"""walk""",2192401
"""sit""",1991919
"""stand""",1851492
"""bike""",1845557
"""stairsup""",1782010
"""stairsdown""",1615896


In [9]:
def create_sliding_window(df, windows_size = 128, step_size = 64):
    X_list = []
    y_list = []
    meta_list = []

    clean_df = df.filter(
        (pl.col("gt").is_not_null()) & (pl.col("gt") != "null")
    )

    groups = clean_df.group_by(['User','Device','gt'])

    for (user,device,activity),group_data in groups:

        sensors = group_data.select(['x','y','z']).to_numpy()

        n_rows = len(sensors)
        if n_rows < windows_size:
            continue

        for start in range(0, n_rows - windows_size + 1, step_size):
            end = start + windows_size

            window = sensors[start : end]

            X_list.append(window)
            y_list.append(activity)
            meta_list.append((user,device))

    return np.array(X_list),np.array(y_list),np.array(meta_list)

print("Loading data slice...")
test_df = pl.read_csv(data_path,n_rows=500000)

print("Slicing into 2.5-second windows...")
X,y,meta = create_sliding_window(test_df)

print(f"Success! Created {X.shape[0]} independent clips.")
print(f"Shape of X (Features): {X.shape} -> (Samples, Time Steps, Channels)")
print(f"Shape of y (Labels):   {y.shape} -> (Samples,)")

Loading data slice...
Slicing into 2.5-second windows...
Success! Created 6987 independent clips.
Shape of X (Features): (6987, 128, 3) -> (Samples, Time Steps, Channels)
Shape of y (Labels):   (6987,) -> (Samples,)


In [11]:
def extract_tabular_features(X_tensor):
    means = np.mean(X_tensor, axis=1)
    stds = np.std(X_tensor, axis=1)
    maxs = np.max(X_tensor, axis=1)
    mins = np.min(X_tensor, axis=1)

    X_tabular = np.hstack([means,stds,maxs,mins])

    channels = ['x','y','z']
    stats = ['mean','std','max','min']
    col_names = [f"{ch}_{st}" for st in stats for ch in channels]

    return pd.DataFrame(X_tabular,columns=col_names)

print("Extracting statistical features...")
X_tab = extract_tabular_features(X)
print(f"Original 3D shape: {X.shape}")
print(f"New Tabular shape: {X_tab.shape} -> (Samples, Features)")
display(X_tab.head())

Extracting statistical features...
Original 3D shape: (6987, 128, 3)
New Tabular shape: (6987, 12) -> (Samples, Features)


,x_mean,y_mean,z_mean,x_std,y_std,z_std,x_max,y_max,z_max,x_min,y_min,z_min
0,-5.775460,0.941105,7.819897,0.091604,0.095064,0.063118,-5.621201,1.090790,7.941147,-5.943741,0.744446,7.676926
1,-5.692695,0.980205,7.876347,0.040911,0.056081,0.042419,-5.552170,1.090790,7.991135,-5.785446,0.844421,7.788803
2,-5.695736,0.926228,7.885813,0.042436,0.049765,0.039508,-5.552170,1.055084,7.991135,-5.815201,0.798004,7.799515
3,-5.719372,0.910188,7.876124,0.039466,0.045031,0.047355,-5.643814,1.018188,8.010178,-5.828293,0.798004,7.732864
4,-5.747583,0.905902,7.850488,0.042936,0.031699,0.052156,-5.650955,1.003906,8.010178,-5.872330,0.852753,7.728104


In [15]:
print("Loading the full 13M row dataset...")
full_df = pl.read_csv(data_path)

print("Slicing into 2.5-second windows (this may take a minute)...")
X_full,y_full,meta_full = create_sliding_window(full_df,windows_size=128,step_size=64)
print(f"Created {X_full.shape[0]} total clips.")

print("Extracting statistical features...")
X_tab_full = extract_tabular_features(X_full)

print("Performing Group-Aware Split...")
user_groups_full = meta_full[:, 0]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X_tab_full, y_full, groups=user_groups_full))

X_train = X_tab_full.iloc[train_idx]
X_test = X_tab_full.iloc[test_idx]
y_train = y_full[train_idx]
y_test = y_full[test_idx]

train_users = np.unique(user_groups_full[train_idx])
test_users = np.unique(user_groups_full[test_idx])

print(f"\nFinal Training set size: {len(X_train)} samples")
print(f"Final Testing set size:  {len(X_test)} samples")
print(f"Users in Training set: {train_users}")
print(f"Users in Testing set:  {test_users}")

Loading the full 13M row dataset...
Slicing into 2.5-second windows (this may take a minute)...
Created 175611 total clips.
Extracting statistical features...
Performing Group-Aware Split...

Final Training set size: 136015 samples
Final Testing set size:  39596 samples
Users in Training set: ['a' 'c' 'd' 'e' 'f' 'g' 'i']
Users in Testing set:  ['b' 'h']


In [18]:
print("Training the Random Forest baseline (this might take a minute or two)...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train,y_train)
print("Predicting on the 2 unseen test users ('b' and 'h')...")
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nBaseline Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

Training the Random Forest baseline (this might take a minute or two)...
Predicting on the 2 unseen test users ('b' and 'h')...

Baseline Accuracy: 68.16%

Detailed Report:
              precision    recall  f1-score   support

        bike       0.81      0.75      0.78      7152
         sit       0.91      0.77      0.83      6841
  stairsdown       0.49      0.67      0.57      5866
    stairsup       0.50      0.49      0.50      6401
       stand       0.83      0.83      0.83      5891
        walk       0.63      0.59      0.61      7445

    accuracy                           0.68     39596
   macro avg       0.69      0.68      0.69     39596
weighted avg       0.70      0.68      0.69     39596

